# Session 4 — Building and Automating Machine Learning Models Using AutoML with Vertex AI

**Goal:** train a tabular classifier without hand-picking an algorithm or
hyperparameters, using Google Cloud's **Vertex AI AutoML** — upload data, kick off a
managed search over model architectures, and deploy the winner to an endpoint.

## What "AutoML" automates

Sessions 1-3 assumed you already knew which model to train (`LogisticRegression`).
AutoML flips that: you hand it a labeled table and a target column, and a managed
service searches over model families, feature transformations, and hyperparameters
for you, then gives you a deployable model. It trades control for speed — good for a
fast baseline or for teams without dedicated ML engineers on every project.

## Prerequisites

This session needs a **Google Cloud project** with billing enabled and the Vertex AI
API turned on — not available in this sandbox, so the cells below are complete,
correct reference code, not executed here.

```bash
pip install google-cloud-aiplatform
gcloud auth application-default login
```

In [ ]:
PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
BUCKET_URI = "gs://your-bucket-name"

## Step 1 — Initialize the Vertex AI SDK

In [ ]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

## Step 2 — Upload a tabular dataset

Vertex AI needs your training data as a managed `Dataset` resource, either from a CSV
in Cloud Storage or a BigQuery table. Here we point at a CSV already uploaded to a
bucket (e.g. the Session 1 `iris`-style table, or the heart disease dataset used
elsewhere in this course, exported to CSV and uploaded with `gsutil cp`).

In [ ]:
dataset = aiplatform.TabularDataset.create(
    display_name="heart-disease-automl",
    gcs_source=[f"{BUCKET_URI}/heart_disease_cleaned.csv"],
)
print(f"Dataset resource name: {dataset.resource_name}")

## Step 3 — Configure and launch the AutoML training job

`optimization_objective` tells AutoML what to optimize for (here, ROC-AUC on a
binary target); `budget_milli_node_hours` caps compute spend — AutoML training is
billed by compute time, not a flat fee, so this budget matters.

In [ ]:
job = aiplatform.AutoMLTabularTrainingJob(
    display_name="heart-disease-automl-job",
    optimization_prediction_type="classification",
    optimization_objective="maximize-au-roc",
)

model = job.run(
    dataset=dataset,
    target_column="target",
    training_fraction_split=0.7,
    validation_fraction_split=0.15,
    test_fraction_split=0.15,
    budget_milli_node_hours=1000,   # 1 node-hour; smallest practical budget for a demo
    model_display_name="heart-disease-automl-model",
    disable_early_stopping=False,
)
print(f"Trained model resource name: {model.resource_name}")

## Step 4 — Inspect the model's evaluation metrics

Vertex AI computes standard evaluation metrics automatically (AUC, precision-recall
curve, confusion matrix, feature importance) — no need to write your own evaluation
code, unlike the manual `accuracy_score` calls in Sessions 1 and 3.

In [ ]:
evaluations = list(model.list_model_evaluations())
for evaluation in evaluations:
    metrics = evaluation.metrics
    print(f"AU ROC   : {metrics.get('auRoc')}")
    print(f"AU PRC   : {metrics.get('auPrc')}")
    print(f"Log loss : {metrics.get('logLoss')}")

## Step 5 — Deploy the winning model to an endpoint

An `Endpoint` is a managed, autoscaling REST service — the AutoML equivalent of the
Flask/FastAPI deployment built by hand in Session 7.

In [ ]:
endpoint = model.deploy(
    machine_type="n1-standard-4",
    min_replica_count=1,
    max_replica_count=1,
)
print(f"Endpoint deployed: {endpoint.resource_name}")

## Step 6 — Get a prediction

In [ ]:
instance = {
    "age": "58", "sex": "1", "cp": "0", "trestbps": "128", "chol": "216",
    "fbs": "0", "restecg": "0", "thalach": "131", "exang": "1",
    "oldpeak": "2.2", "slope": "1", "ca": "3", "thal": "3",
}
prediction = endpoint.predict(instances=[instance])
print(prediction)

## Step 7 — Clean up

AutoML endpoints bill per hour while deployed, regardless of traffic — undeploy when
you're done experimenting.

In [ ]:
endpoint.undeploy_all()
endpoint.delete()
print("Endpoint undeployed and deleted -- billing stopped.")

## What to try next

* Compare the AutoML model's AUC against the hand-picked models from Session 1's
  MLflow comparison on the same dataset — AutoML often wins on tabular data with
  little tuning effort, but costs real money per training run.
* Session 9 repeats this "AutoML → managed endpoint" pattern on AWS SageMaker
  Autopilot instead — useful for seeing how the same idea differs across clouds.
* Session 25 shows a fully local, free, open-source AutoML alternative (FLAML) for
  when a managed cloud service isn't available or justified.